In [ ]:
# E23 P6b: EAGLE-Corp/Qwen3.8-27B-EAGLE4 drafter, pinned revision, sha-checked, flat output (internet ON)
import hashlib, json, os, re, time, urllib.request
from pathlib import Path
T0 = time.time()
for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HF_HUB_OFFLINE"):
    os.environ.pop(k, None)
os.environ.update(HF_HOME="/tmp/e23p6b_hf", HF_XET_CHUNK_CACHE_SIZE_BYTES="0", HF_HUB_DISABLE_PROGRESS_BARS="1")
REPO, REV = "EAGLE-Corp/Qwen3.8-27B-EAGLE4", "f58a5985223040552ad404fba98f0dd27f5f676a"
SKIP = {".gitattributes"}
W = Path("/kaggle/working")
R = {"repo": REPO, "revision": REV, "files": {}, "errors": []}
try:
    tree = json.load(urllib.request.urlopen(f"https://huggingface.co/api/models/{REPO}/tree/{REV}?recursive=true", timeout=60))
    want = {e["path"]: (e.get("size"), (e.get("lfs") or {}).get("oid")) for e in tree if e.get("type") == "file" and e["path"] not in SKIP}
    R["remote"] = want
    from huggingface_hub import hf_hub_download
    for n, (size, oid) in sorted(want.items(), key=lambda kv: kv[1][0] or 0):
        t = time.time()
        p = Path(hf_hub_download(REPO, n, revision=REV, local_dir=str(W)))
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for b in iter(lambda: f.read(8 << 20), b""):
                h.update(b)
        rec = {"size": p.stat().st_size, "sha256": h.hexdigest(), "sec": round(time.time() - t, 1)}
        rec["ok"] = rec["size"] == size and (oid is None or rec["sha256"] == oid)
        R["files"][n] = rec
        print(n, rec, flush=True)
        if not rec["ok"]:
            R["errors"].append(f"{n}: size/sha mismatch")
    cfg = json.loads((W / "config.json").read_text())
    R["config"] = {k: cfg.get(k) for k in ("architectures", "model_type", "hidden_size", "num_hidden_layers", "vocab_size", "torch_dtype")}
except Exception as e:
    R["errors"].append(f"fatal: {e!r}")
finally:
    import shutil
    shutil.rmtree(W / ".cache", ignore_errors=True)
    bad = sorted(str(p.relative_to(W)) for p in W.rglob("*") if p.is_dir() or (p.is_file() and (
        len(p.relative_to(W).parts) > 1 or p.name.startswith(".") or re.search(r"[^A-Za-z0-9._-]", p.name))))
    R["bad_output_names"] = bad
    if bad:
        R["errors"].append(f"output not flat / safe: {bad}")
    R["all_ok"] = not R["errors"] and bool(R["files"]) and all(v["ok"] for v in R["files"].values())
    R["sec"] = round(time.time() - T0)
    (W / "p6b_report.json").write_text(json.dumps(R, indent=1))
    print(json.dumps({k: R.get(k) for k in ("config", "errors", "bad_output_names")}), flush=True)
    print("ALL OK" if R["all_ok"] else "NOT OK", "|", R["sec"], "s", flush=True)
